# ERA V5 · Session 12 | ZeRO across 32 Virtual GPUs

**Real local PyTorch computation + simulated distributed ownership** · reproducible capstone preparation.

**Important:** This notebook uses **32 logical ranks**, not 32 physical CUDA GPUs. It trains an actual small model, partitions state/optimizer and materializes ZeRO-3 layers for forward/backward. The memory and communication figures are ideal analytical models, not measured accelerator VRAM/network. CPU benchmark times are **not GPU speedups**.

The learning goal is to demonstrate why **ZeRO-0 → 1 → 2 → 3** partitions successively optimizer state, gradients and parameters while maintaining the same mathematical training trajectory.

## Audit assumptions before running

- `N=32` virtual ranks; 2 supervised samples per rank → 64 samples/optimizer step.
- Network-free `torch.cat` emulates layer parameter all-gather; Stage-3 autograd backward **rematerializes** parameters by layer.
- Real PyTorch compute uses **FP64** to detect mathematical errors; separate 16-byte/parameter memory model assumes BF16 weights/gradients and FP32 master/moments.
- Illustrative communication uses ring **payload bytes per rank** and does not assume physical network performance.
- Cited originals: [ZeRO paper (2019)](https://arxiv.org/abs/1910.02054), [official ZeRO stage guide](https://deepspeed.readthedocs.io/en/stable/zero3.html).
- Transcript source: *ERA V5 Session 12* (September 12, 2026), supplied by the assignment author. A separate lesson-notes file was not available in the attachments.

In [ ]:
import os, sys, platform, torch
from pathlib import Path
# Small matrix multiplications avoid massive thread-pool overhead on CPU.
torch.set_num_threads(1)
print('Torch:', torch.__version__, 'Python:', platform.python_version())
print('CUDA is available:', torch.cuda.is_available())
print('Physical GPU count USED in this notebook:', 0)
print('Virtual logical GPU ranks:', 32)
assert torch.__version__
Path('src').mkdir(exist_ok=True)
Path('tests').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)

## Executable implementation — embedded, so Colab runs without a GitHub clone

The next cell writes **the exact same source** as the accompanying `src/virtual_zero.py` and `run_experiment.py` files. Source is included here rather than downloaded from any external URL. You can inspect all logic by expanding the cell.

In [ ]:
from pathlib import Path
Path('src/virtual_zero.py').write_text('"""32 logical-rank ZeRO teaching simulator, intentionally NOT a distributed GPU runtime.\n\nUses real PyTorch forward/backward on CPU, simulated ownership/collective transfers,\nreal sharded Adam updates, and analytical BF16/FP32 memory & ring-traffic estimates.\n"""\nfrom __future__ import annotations\nimport dataclasses\nimport time\nfrom typing import List\nimport torch\nimport torch.nn.functional as F\n\nWORLD_SIZE = 32\nINPUT_DIM = 16\nHIDDEN_DIM = 32\nCLASSES = 8\nLAYER_ELEMENTS = (HIDDEN_DIM * INPUT_DIM + HIDDEN_DIM, CLASSES * HIDDEN_DIM + CLASSES)\nNUM_PARAMS = sum(LAYER_ELEMENTS)\nLR = 0.008\nBETA1, BETA2, EPS = .9, .999, 1e-8\n\n@dataclasses.dataclass\nclass Rank:\n    id: int\n    left: int\n    right: int\n    params: torch.Tensor\n    first_moment: torch.Tensor\n    second_moment: torch.Tensor\n    grad: torch.Tensor | None = None\n    owner_gradient: torch.Tensor | None = None\n\n\ndef partitions(count: int, world: int = WORLD_SIZE):\n    return [(i * count // world, (i+1) * count // world) for i in range(world)]\n\n\ndef initial_weights(seed: int = 20260912):\n    gen = torch.Generator().manual_seed(seed)\n    return (torch.randn(NUM_PARAMS, generator=gen, dtype=torch.float64) * .055).detach()\n\n\ndef rank_data(seed: int = 312, world: int = WORLD_SIZE, batch_per_rank: int = 2):\n    """Fixed deterministic supervised toy data; no external corpus claims."""\n    g = torch.Generator().manual_seed(seed)\n    x = torch.randn(world, batch_per_rank, INPUT_DIM, generator=g, dtype=torch.float64)\n    teacher = torch.randn(INPUT_DIM, CLASSES, generator=g, dtype=torch.float64)\n    y = (x @ teacher).argmax(-1)\n    return [(x[i], y[i]) for i in range(world)]\n\n\ndef linear_pieces(ranks: list[Rank], absolute_start: int, length: int):\n    """The correct sliced owner fragments, without constructing an entire flat model."""\n    end = absolute_start + length\n    pieces = []\n    for r in ranks:\n        lo = max(r.left, absolute_start)\n        hi = min(r.right, end)\n        if lo < hi:\n            pieces.append(r.params[lo-r.left:hi-r.left])\n    assert sum(p.numel() for p in pieces) == length\n    return pieces\n\n\nclass JustInTimeShardedLinear(torch.autograd.Function):\n    """Gather a layer in forward and rematerialize it in backward, then shard its gradient.\n\n    This is **local torch.cat**, not a hardware or NCCL collective. Backward saves only\n    input and owner views, not the gathered parameter tensor.\n    """\n    @staticmethod\n    def forward(ctx, x, out_dim: int, in_dim: int, *pieces):\n        full = torch.cat(pieces, dim=0)\n        weight = full[:out_dim*in_dim].reshape(out_dim, in_dim)\n        bias = full[out_dim*in_dim:]\n        ctx.save_for_backward(x, *pieces)\n        ctx.dims = (out_dim, in_dim)\n        ctx.sizes = [p.numel() for p in pieces]\n        return F.linear(x, weight, bias)\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        x, *pieces = ctx.saved_tensors\n        out_dim, in_dim = ctx.dims\n        # Second parameter materialization during backward: ZeRO-3-like behavior.\n        full = torch.cat(pieces, dim=0)\n        w = full[:out_dim*in_dim].reshape(out_dim, in_dim)\n        grad_x = grad_output @ w\n        grad_w = grad_output.transpose(-1,-2) @ x\n        grad_b = grad_output.sum(dim=0)\n        grad_flat = torch.cat((grad_w.reshape(-1), grad_b))\n        return (grad_x, None, None, *grad_flat.split(ctx.sizes))\n\n\ndef full_forward(x, flat):\n    first, second = LAYER_ELEMENTS\n    w1 = flat[:HIDDEN_DIM*INPUT_DIM].reshape(HIDDEN_DIM, INPUT_DIM)\n    b1 = flat[HIDDEN_DIM*INPUT_DIM:first]\n    w2 = flat[first:first+CLASSES*HIDDEN_DIM].reshape(CLASSES, HIDDEN_DIM)\n    b2 = flat[first+CLASSES*HIDDEN_DIM:]\n    return F.linear(F.relu(F.linear(x,w1,b1)), w2, b2)\n\n\ndef stage3_forward(x, ranks):\n    first, second = LAYER_ELEMENTS\n    p1 = linear_pieces(ranks, 0, first)\n    h = F.relu(JustInTimeShardedLinear.apply(x, HIDDEN_DIM, INPUT_DIM, *p1))\n    p2 = linear_pieces(ranks, first, second)\n    return JustInTimeShardedLinear.apply(h, CLASSES, HIDDEN_DIM, *p2)\n\n\ndef initialize(stage: int, start: torch.Tensor, world: int = WORLD_SIZE):\n    assert stage in (0, 1, 2, 3)\n    result=[]\n    for i,(lo,hi) in enumerate(partitions(start.numel(),world)):\n        # Keep model parameter full except stage 3.\n        p = start[lo:hi].clone() if stage==3 else start.clone()\n        # Optimizer states stay full only for baseline (stage 0).\n        m = torch.zeros_like(start if stage==0 else start[lo:hi])\n        v = torch.zeros_like(m)\n        p.requires_grad_(stage==3)\n        result.append(Rank(i,lo,hi,p,m,v))\n    return result\n\n\ndef adam_inplace(p:torch.Tensor,m:torch.Tensor,v:torch.Tensor,g:torch.Tensor,step:int):\n    """Adam, with no weight decay, identical update for every stage."""\n    with torch.no_grad():\n        m.mul_(BETA1).add_(g,alpha=1-BETA1)\n        v.mul_(BETA2).addcmul_(g,g,value=1-BETA2)\n        update = (m / (1-BETA1**step)) / ((v/(1-BETA2**step)).sqrt()+EPS)\n        p.add_(update,alpha=-LR)\n\n\ndef reconstruct(stage:int,ranks:list[Rank]):\n    return torch.cat([r.params.detach() for r in ranks]) if stage==3 else ranks[0].params.detach().clone()\n\n\ndef optimizer_step(stage:int,ranks:list[Rank],step:int):\n    if stage==0:\n        for r in ranks:\n            assert r.grad is not None\n            adam_inplace(r.params,r.first_moment,r.second_moment,r.grad,step)\n    else:\n        for r in ranks:\n            assert r.grad is not None\n            if stage==3:\n                # Parameters, gradients, optimizer moments owned only by this rank.\n                adam_inplace(r.params,r.first_moment,r.second_moment,r.grad,step)\n            else:\n                # Only shard owners update; afterward all-gather updated weights.\n                segment = r.params[r.left:r.right]\n                adam_inplace(segment,r.first_moment,r.second_moment,\n                             r.owner_gradient if stage==1 else r.grad,step)\n        if stage in (1,2):\n            assembled = torch.cat([r.params[r.left:r.right] for r in ranks]).detach()\n            for r in ranks:\n                r.params.copy_(assembled)\n\n\ndef execute(stage:int,steps:int=4,world:int=WORLD_SIZE, verbose:bool=False):\n    assert world == WORLD_SIZE, \'This assignment deliberately fixes 32 virtual ranks.\'\n    ranks = initialize(stage, initial_weights(), world)\n    data=rank_data(world=world)\n    losses=[]; step_ms=[]\n    with torch.no_grad():\n        _=full_forward(data[0][0], initial_weights())\n    for step in range(1,steps+1):\n        t0=time.perf_counter()\n        if stage==3:\n            for r in ranks:\n                r.params.grad = None\n            batch_losses=[]\n            for x,y in data:\n                score = stage3_forward(x,ranks)\n                loss = F.cross_entropy(score,y)\n                (loss / world).backward()\n                batch_losses.append(loss.detach().item())\n            # Each owner has its reduced/scattered gradient accumulated on its own shard.\n            for r in ranks: r.grad = r.params.grad.detach().clone()\n        else:\n            local_full_grads=[]; batch_losses=[]\n            for rank,(x,y) in zip(ranks,data):\n                w = rank.params.clone().detach().requires_grad_(True)\n                score = full_forward(x,w)\n                loss = F.cross_entropy(score,y)\n                loss.backward()\n                local_full_grads.append(w.grad.detach().clone())\n                batch_losses.append(loss.detach().item())\n            if stage==0:\n                average = torch.stack(local_full_grads).mean(0)\n                for r in ranks: r.grad = average.clone() # all-reduce\n            elif stage==1:\n                # Each rank retains its *full local gradient*; reduce-scatter the\n                # synchronized gradient only to the optimizer shard\'s owner.\n                for r,g in zip(ranks,local_full_grads):\n                    r.grad = g.clone()\n                    r.owner_gradient = sum((q[r.left:r.right] for q in local_full_grads),\n                                           torch.zeros(r.right-r.left,dtype=torch.float64))/world\n            elif stage==2:\n                # Reduce-scatter directly into owner buffers, without retaining full reduced grads.\n                for r in ranks:\n                    r.grad = sum((g[r.left:r.right] for g in local_full_grads),\n                                 torch.zeros(r.right-r.left,dtype=torch.float64)) / world\n        losses.append(sum(batch_losses)/world)\n        optimizer_step(stage,ranks,step)\n        step_ms.append((time.perf_counter()-t0)*1000)\n        if verbose: print(f\'  step {step}: pre-update global CE={losses[-1]:.10f}, elapsed_cpu_ms={step_ms[-1]:.2f}\')\n        # Exact agreement of full model copies in baseline / stages 1/2.\n        if stage!=3:\n            assert all(torch.equal(r.params,ranks[0].params) for r in ranks)\n    return dict(stage=stage, world=world, losses=losses, final=reconstruct(stage,ranks),\n                mean_cpu_step_ms=sum(step_ms)/len(step_ms), cpu_step_ms=step_ms,\n                rank_state_elements=[dict(rank=r.id,weight=r.params.numel(),\n                    gradient=r.grad.numel(),moments=r.first_moment.numel()+r.second_moment.numel()) for r in ranks])\n\n\ndef resident_bytes_per_parameter(stage, world=32):\n    # Ideal accounting: 2B low precision parameters, 2B low precision gradients,\n    # 4B fp32 master weights + 4B momentum + 4B variance = 12B optimizer state.\n    return ({0:16,1:4+12/world,2:2+14/world,3:16/world})[stage]\n\n\ndef memory_table(params=NUM_PARAMS,world=WORLD_SIZE):\n    # \'comm\' uses ring volume per rank in units of S = params * 2B,\n    # not wall-clock transfer time, and excludes latency / overlap / temporary buffers.\n    factor=(world-1)/world\n    out=[]\n    for stage in range(4):\n        p= params * resident_bytes_per_parameter(stage, world)\n        # Illustrative, optimally staged ring collective: stage0 AR (2S),\n        # stage1/2 RS + AG (2S), stage3 RS + forward AG + backward AG (3S).\n        units = factor*(3 if stage==3 else 2)\n        comm = params*2*units\n        out.append(dict(stage=stage,resident_bytes_per_rank=p,\n                    resident_MiB_per_rank=p/2**20,\n                    resident_fraction_of_dp=p/(params*16),\n                    modeled_ring_bytes_per_rank=comm,\n                    modeled_ring_MiB_per_rank=comm/2**20,\n                    communication_S_units=units,\n                    # Dense forward/backward computation is not sharded across data ranks.\n                    forward_backward_relative_flops_per_rank=1.0,\n                    optimizer_math_elements_per_rank=params if stage==0 else params/world,\n                    optimizer_math_fraction=1.0 if stage==0 else 1/world))\n    return out\n\n\ndef stage3_extra_buffer_bytes(layer_sizes=LAYER_ELEMENTS):\n    # At least one gathered BF16 layer parameter buffer (excluding fragmented grads & caching).\n    return max(layer_sizes)*2\n\n\ndef projected_state_for_30b(world=32):\n    return memory_table(params=30_000_000_000, world=world)\n\n\ndef validate():\n    assert len(partitions(NUM_PARAMS)) == WORLD_SIZE\n    ranges=partitions(NUM_PARAMS)\n    assert ranges[0][0]==0 and ranges[-1][1]==NUM_PARAMS\n    assert all(ranges[i][1]==ranges[i+1][0] for i in range(WORLD_SIZE-1))\n    assert sum(end-start for start,end in ranges)==NUM_PARAMS\n    assert max(end-start for start,end in ranges)-min(end-start for start,end in ranges)<=1\n    x=rank_data()[0][0]\n    shard=initialize(3,initial_weights())\n    with torch.no_grad():\n        a=full_forward(x,initial_weights())\n        b=stage3_forward(x,shard)\n    assert torch.allclose(a,b,atol=1e-14,rtol=1e-14)\n    assert sum(r.params.numel() for r in shard)==NUM_PARAMS\n    assert all(r.params.numel() < NUM_PARAMS for r in shard)\n    for n in (8,32):\n        assert resident_bytes_per_parameter(0,n)==16\n        assert resident_bytes_per_parameter(3,n)==16/n\n        assert resident_bytes_per_parameter(1,n)==4+12/n\n        assert resident_bytes_per_parameter(2,n)==2+14/n\n    results=[execute(stage=s,steps=4) for s in range(4)]\n    ref=results[0]\n    for r in results[1:]:\n        max_diff=(ref[\'final\']-r[\'final\']).abs().max().item()\n        max_loss_diff=max(abs(a-b) for a,b in zip(ref[\'losses\'],r[\'losses\']))\n        assert max_diff<1e-12,(r[\'stage\'],max_diff)\n        assert max_loss_diff<1e-12,(r[\'stage\'],max_loss_diff)\n        r[\'max_parameter_error_vs_baseline\']=max_diff\n        r[\'max_loss_error_vs_baseline\']=max_loss_diff\n    ref[\'max_parameter_error_vs_baseline\']=0.0\n    ref[\'max_loss_error_vs_baseline\']=0.0\n    assert ref[\'losses\'][-1] < ref[\'losses\'][0], ref[\'losses\']\n    return results\n\nif __name__ == \'__main__\':\n    print(\'ENV: torch\',torch.__version__,\'CUDA\',torch.cuda.is_available(),\'CPU threads\',torch.get_num_threads())\n    print(\'Model params\',NUM_PARAMS,\'32 virtual ranks\')\n    results=validate()\n    for r in results:\n        print(r[\'stage\'],[\'%.9f\'%v for v in r[\'losses\']],\n              \'err\',r[\'max_parameter_error_vs_baseline\'],\'mean CPU ms\',round(r[\'mean_cpu_step_ms\'],2))\n    for x in memory_table(): print(x)\n')
Path('run_experiment.py').write_text('"""Reproduce notebook results and export plots, CSV and JSON in repository."""\nimport json,sys,os,csv,platform\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\nROOT=Path(__file__).resolve().parent\nsys.path.insert(0,str(ROOT/\'src\'))\nimport virtual_zero as z\n\ndef produce(base=ROOT):\n    torch.set_num_threads(1)\n    (base/\'figures\').mkdir(exist_ok=True,parents=True)\n    (base/\'results\').mkdir(exist_ok=True,parents=True)\n    trials=z.validate()\n    compact=[]\n    for t in trials:\n        compact.append({k:v for k,v in t.items() if k not in (\'final\',\'rank_state_elements\')})\n    toy=z.memory_table()\n    large=z.projected_state_for_30b()\n    output=dict(seed_weights=20260912,seed_data=312,world_size=32,real_gpu_count=0,\n        virtual_ranks=32,model=\'2 linear layer classifier with ReLU (synthetic teacher labels)\',\n        model_parameter_count=z.NUM_PARAMS,global_batch=64,batch_per_rank=2,\n        optimizer=\'Adam beta1=.9 beta2=.999 eps=1e-8 LR=.008, fp64 execution\',\n        runtime_torch=torch.__version__,python=platform.python_version(),cpu_threads=1,\n        experimental_results=compact,ideal_mixed_precision_toy=toy,\n        analytical_projection_30b=large,\n        estimated_Z3_minimum_one_gathered_layer_bytes=z.stage3_extra_buffer_bytes(),\n        notes=[\'No GPUs used; local CPU torch.cat models all-gather semantics without networking.\',\n          \'Communication estimates are ideal ring-collective payload per rank, not measured bandwidth/time.\',\n          \'Ideal mixed-precision memory is separate from actual fp64 CPU allocations.\',\n          \'ZeRO3 extra gathered layer and activation allocations are excluded from the persistent-state table.\',\n          \'Forward/backward dense model FLOPs per rank are unchanged, Adam state updates per rank decrease.\'])\n    (base/\'results\'/\'experiment.json\').write_text(json.dumps(output,indent=2))\n    with (base/\'results\'/\'memory_and_communication.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=[\'stage\',\'bytes_per_param_per_rank\',\'30b_state_GiB\',\'30b_ring_GB_per_rank\',\'ring_units_of_fp16_param_bytes\', \'optimizer_elements_per_rank_avg\'])\n        writer.writeheader()\n        for s,t in enumerate(large):\n            writer.writerow(dict(stage=s,bytes_per_param_per_rank=z.resident_bytes_per_parameter(s),\n                **{\'30b_state_GiB\':round(t[\'resident_bytes_per_rank\']/2**30,5),\n                \'30b_ring_GB_per_rank\':round(t[\'modeled_ring_bytes_per_rank\']/1e9,5),\n                \'ring_units_of_fp16_param_bytes\':t[\'communication_S_units\'],\n                \'optimizer_elements_per_rank_avg\':30e9/(1 if s==0 else 32)}))\n    colors=[\'#1f3855\',\'#217da9\',\'#e7ab37\',\'#58b8a6\']\n    fig,axs=plt.subplots(1,2,figsize=(12,4.4))\n    x=np.arange(4)\n    value=[t[\'resident_bytes_per_rank\']/2**30 for t in large]\n    bars=axs[0].bar(x,value,color=colors)\n    axs[0].set_xticks(x,[f\'ZeRO-{s}\' for s in x]);axs[0].set_ylabel(\'Persistent state per virtual GPU (GiB)\')\n    axs[0].set_title(\'30B-parameter example · 32 ranks\')\n    for b,v in zip(bars,value):axs[0].text(b.get_x()+b.get_width()/2,v+4,f\'{v:.1f}\',ha=\'center\',fontsize=10)\n    axs[0].set_ylim(0,max(value)*1.15)\n    val=[t[\'modeled_ring_bytes_per_rank\']/1e9 for t in large]\n    bars=axs[1].bar(x,val,color=colors)\n    axs[1].set_xticks(x,[f\'ZeRO-{s}\' for s in x]);axs[1].set_ylabel(\'Ideal ring payload / rank / step (GB)\')\n    axs[1].set_title(\'Illustrative communication volume · NOT timing\')\n    for b,v in zip(bars,val):axs[1].text(b.get_x()+b.get_width()/2,v+3,f\'{v:.2f}\',ha=\'center\',fontsize=10)\n    axs[1].set_ylim(0,max(val)*1.15)\n    fig.suptitle(\'ZeRO trades persistent state for communication\',fontsize=15,fontweight=\'bold\')\n    fig.tight_layout();fig.savefig(base/\'figures\'/\'state_vs_communication.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    fig,ax=plt.subplots(figsize=(9.8,4.4))\n    for t,c in zip(compact,colors):\n        ax.plot(np.arange(1,len(t[\'losses\'])+1),t[\'losses\'],marker=\'o\',color=c,\n                alpha=.85,linewidth=2,label=f"ZeRO-{t[\'stage\']}")\n    ax.set_title(\'Real PyTorch training · 32 logical ranks · four identical trajectories\')\n    ax.set_xlabel(\'Optimizer step\');ax.set_ylabel(\'Global supervised cross-entropy (pre-update)\')\n    ax.set_xticks(range(1,len(compact[0][\'losses\'])+1));ax.grid(alpha=.2);ax.legend(ncol=4,loc=\'upper right\')\n    fig.tight_layout();fig.savefig(base/\'figures\'/\'loss_equivalence.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    # Exact rank-level state distribution; derived from resident tensors and byte assumptions.\n    per_rank=[]\n    for stage in range(4):\n        row=[]\n        for r in trials[stage][\'rank_state_elements\']:\n            # BF16 W/G are 2 bytes each; 12B optimizer = FP32 master+first+second moment.\n            # There are TWO stored moments in code; ideal accounting includes separate FP32 master.\n            o=(r[\'moments\']//2)*12\n            row.append((r[\'weight\']*2+r[\'gradient\']*2+o)/1024)\n        per_rank.append(row)\n    fig,ax=plt.subplots(figsize=(12,3.8))\n    im=ax.imshow(per_rank,aspect=\'auto\',cmap=\'YlGnBu\')\n    ax.set_yticks(range(4),[f\'ZeRO-{j}\' for j in range(4)])\n    ax.set_xticks(range(0,32,2),[f\'{j}\' for j in range(0,32,2)])\n    ax.set_xlabel(\'Virtual rank (0–31)\')\n    ax.set_title(\'32 ranks · resident training-state distribution (ideal KB)\')\n    fig.colorbar(im,ax=ax,label=\'KiB / rank\',shrink=.8)\n    fig.tight_layout();fig.savefig(base/\'figures\'/\'32_rank_ownership.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    fig,ax=plt.subplots(figsize=(9,4.5))\n    b1=ax.bar(np.arange(4)-0.18,[1,1,1,1],.36,color=\'#217da9\',label=\'Dense fwd+bwd FLOPs per rank (relative)\')\n    b2=ax.bar(np.arange(4)+0.18,[1,1/32,1/32,1/32],.36,color=\'#e7ab37\',label=\'Optimizer element updates per rank (relative)\')\n    ax.set_xticks(range(4),[f\'ZeRO-{s}\' for s in range(4)])\n    ax.set_ylim(0,1.25);ax.set_title(\'Sharding optimizer work ≠ skipping model computation\')\n    ax.legend(fontsize=9);ax.set_ylabel(\'Relative to replicated data parallel baseline\')\n    fig.tight_layout();fig.savefig(base/\'figures\'/\'compute_vs_optimizer.png\',dpi=170,bbox_inches=\'tight\');plt.close(fig)\n\n    print(\'Validation: 32 ranks, ZeRO-0/1/2/3; all tests passed\')\n    print(\'The training losses are identical to displayed precision:\')\n    for t in compact:\n        print(f"  stage {t[\'stage\']}: losses={[round(x,9) for x in t[\'losses\']]} "+\n           f"max_weight_error={t[\'max_parameter_error_vs_baseline\']:.3g}, "+\n           f"mean CPU step={t[\'mean_cpu_step_ms\']:.2f} ms")\n    print(\'Analytical persistent-state example, 30B parameters / 32 ranks:\')\n    for s,t in enumerate(large):\n        print(f"  stage {s}: {z.resident_bytes_per_parameter(s):.4f} bytes/param/rank, "+\n            f"{t[\'resident_bytes_per_rank\']/2**30:.3f} GiB, ring {t[\'modeled_ring_bytes_per_rank\']/1e9:.3f} GB/rank/step")\n    print(\'Important: GPU time, actual GPU allocated VRAM and actual network timings are NOT measured.\')\n    return output\n\nif __name__==\'__main__\':produce()\n')
Path('tests/test_virtual_zero.py').write_text('"""Run with: python -m unittest discover -s tests -v"""\nimport sys\nfrom pathlib import Path\nimport unittest\nimport torch\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'src\'))\nimport virtual_zero as z\n\nclass TestZeRO32(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch.set_num_threads(1)\n        cls.results = z.validate()\n\n    def test_exactly_32_nonoverlapping_shards(self):\n        bounds=z.partitions(z.NUM_PARAMS)\n        self.assertEqual(len(bounds),32)\n        self.assertEqual(bounds[0][0],0)\n        self.assertEqual(bounds[-1][1],z.NUM_PARAMS)\n        self.assertTrue(all(bounds[i][1]==bounds[i+1][0] for i in range(31)))\n\n    def test_identical_model_updates_across_stages(self):\n        ref=self.results[0]\n        for variant in self.results[1:]:\n            self.assertLess((ref[\'final\']-variant[\'final\']).abs().max().item(),1e-12)\n            self.assertLess(max(abs(a-b) for a,b in zip(ref[\'losses\'],variant[\'losses\'])),1e-12)\n\n    def test_real_forward_and_backward_training(self):\n        self.assertLess(self.results[0][\'losses\'][-1],self.results[0][\'losses\'][0])\n        self.assertEqual(len(self.results[0][\'losses\']),4)\n\n    def test_resident_optimizer_and_gradient_ownership(self):\n        for stage, expected_weights_full, expected_grad_full in [(0,True,True),(1,True,True),(2,True,False),(3,False,False)]:\n            v=self.results[stage]\n            for rank in v[\'rank_state_elements\']:\n                self.assertEqual(rank[\'weight\']==z.NUM_PARAMS,expected_weights_full)\n                self.assertEqual(rank[\'gradient\']==z.NUM_PARAMS,expected_grad_full)\n                target=2*z.NUM_PARAMS if stage==0 else 2*(z.partitions(z.NUM_PARAMS)[rank[\'rank\']][1]-z.partitions(z.NUM_PARAMS)[rank[\'rank\']][0])\n                self.assertEqual(rank[\'moments\'],target)\n\n    def test_ideal_bytes_formulas(self):\n        self.assertEqual([z.resident_bytes_per_parameter(x,32) for x in range(4)],\n                         [16,4.375,2.4375,.5])\n        self.assertEqual(z.stage3_extra_buffer_bytes(),z.LAYER_ELEMENTS[0]*2)\n\n    def test_collective_traffic_assumptions(self):\n        t=z.memory_table()\n        self.assertEqual(t[0][\'modeled_ring_bytes_per_rank\'],t[1][\'modeled_ring_bytes_per_rank\'])\n        self.assertEqual(t[1][\'modeled_ring_bytes_per_rank\'],t[2][\'modeled_ring_bytes_per_rank\'])\n        self.assertEqual(t[3][\'modeled_ring_bytes_per_rank\']/t[0][\'modeled_ring_bytes_per_rank\'],1.5)\n\n    def test_native_torch_adam_step(self):\n        # Independently validate hand-coded Adam once against actual torch.optim.Adam.\n        start=z.initial_weights()\n        grad=torch.linspace(-.4,.6,z.NUM_PARAMS,dtype=torch.float64)\n        native=torch.nn.Parameter(start.clone())\n        optimizer=torch.optim.Adam([native],lr=z.LR,betas=(z.BETA1,z.BETA2),eps=z.EPS,weight_decay=0.)\n        native.grad=grad.clone()\n        optimizer.step()\n        ours=start.clone()\n        z.adam_inplace(ours,torch.zeros_like(ours),torch.zeros_like(ours),grad,1)\n        self.assertLess((native.detach()-ours).abs().max().item(),2e-13)\n\nif __name__==\'__main__\': unittest.main()\n')
print('Canonical source files materialized locally; no external downloads.')

In [ ]:
sys.path.insert(0,str(Path('src').resolve()))
import virtual_zero as z
print('Actual model: input', z.INPUT_DIM, '→ hidden', z.HIDDEN_DIM, '→ classes', z.CLASSES)
print('Parameter shapes: W1', (z.HIDDEN_DIM,z.INPUT_DIM), 'b1',(z.HIDDEN_DIM,),
      'W2',(z.CLASSES,z.HIDDEN_DIM),'b2',(z.CLASSES,))
print('Trainable scalar parameters:',z.NUM_PARAMS)
print('One local input shape:',tuple(z.rank_data()[0][0].shape), '[samples_per_rank, features]')
print('One local label shape:',tuple(z.rank_data()[0][1].shape), '[samples_per_rank]')
print('Ranks:',len(z.partitions(z.NUM_PARAMS)), '| partition samples:',
      z.partitions(z.NUM_PARAMS)[:3], '...', z.partitions(z.NUM_PARAMS)[-2:])
assert len(z.partitions(z.NUM_PARAMS))==32

## Stage-by-stage resident-state ownership

| Mode | Model parameter replica | Full local gradients | Optimizer owner state | Parameter all-gathers during layer execution |
|---|---|---|---|---|
| ZeRO-0 | Full on all 32 | Full on all 32 | Full on all 32 | None |
| ZeRO-1 | Full on all 32 | Full on all 32 | Only owned shard | Updated weights after step |
| ZeRO-2 | Full on all 32 | Only reduced gradient shard retained | Only owned shard | Updated weights after step |
| ZeRO-3 | Only owned parameter shard | Only reduced gradient shard | Only owned shard | Forwards **and** backwards |

Each rank computes its own full dense model on *different training samples*. Ownership differs, **math must not change**.

## 32-rank real forward/backward/Adam experiment

The following call does actual tensor operations, checks every stage against ZeRO-0, writes machine-readable JSON and generates four figures. Fixed 4-step run, global batch 64.

In [ ]:
from run_experiment import produce
results = produce(Path('.'))
assert results['world_size']==32
assert results['virtual_ranks']==32
assert results['real_gpu_count']==0
assert len(results['experimental_results'])==4
for e in results['experimental_results'][1:]:
    assert e['max_parameter_error_vs_baseline']<1e-12
    assert e['max_loss_error_vs_baseline']<1e-12
print('PASS: all four modes converge along the same numerically verified gradient/optimizer path.')

## Visual evidence: memory and communications

The chart below projects *only ideal persistent model training state* for a **30B-parameter hypothetical model**. It does not say the 30B model was executed or fits into memory in practice: activation, transient gather and other buffers are omitted.

For a 32-rank ideal BF16/FP32 mixed-precision Adam setup, state bytes/parameter/rank = `16`, `4 + 12/32`, `2 + 14/32`, `16/32` respectively.

In [ ]:
from IPython.display import display, Image
display(Image(filename='figures/state_vs_communication.png'))
print('Temporary ZeRO-3 one-largest-layer BF16 gather for THIS toy:',z.stage3_extra_buffer_bytes(),'bytes (minimum example; not total peak).')

## What happens to computation?

The per-rank **dense forward and backward FLOP count stays unchanged**. The number of local Adam element updates falls approximately 32-fold from stage 0 to stages 1–3, while the **cluster-wide** number of elements updated remains P. Extra gather/buffer traffic is introduced in stage 3.

In [ ]:
display(Image(filename='figures/compute_vs_optimizer.png'))

## Does sharding change the numerical training result?

No (apart from floating-point rounding/layout). All 32 local data partitions are included in the same global gradient; identical initialization and Adam rules produce the same trajectory. The following curves are actual measured loss values, NOT synthetic chart coordinates.

In [ ]:
display(Image(filename='figures/loss_equivalence.png'))

## Are all 32 ranks represented, including uneven shards?

808 parameters are not divisible by 32: a correct partition divides them into ranks owning **25 or 26 elements**, without overlap or gaps. The plot uses the actual state tensor ownership counts in the executed example.

In [ ]:
display(Image(filename='figures/32_rank_ownership.png'))

## An independent audit: 7 executable tests

These tests check owner partition exhaustiveness; exact stage-update parity; training loss decreases; stage-dependent resident tensors; memory formula; communication model assumptions; and our Adam update versus native `torch.optim.Adam`. The notebook raises an error if any assertion fails.

In [ ]:
import unittest
loader=unittest.defaultTestLoader
suite=loader.discover('tests',pattern='test_*.py')
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful() and result.testsRun==7, 'Submission is blocked until seven tests pass'
print('Verified',result.testsRun,'tests, failures:',len(result.failures),'errors:',len(result.errors))

## Five quick questions to demonstrate *human* understanding

1. **What changes from ZeRO-0 to ZeRO-1?** Shard optimizer master weights and Adam moments; still replicate compute weights and local gradients.
2. **What changes from ZeRO-1 to ZeRO-2?** Retain only the owner's reduced gradient shards; local backward may still use temporary gradient buffers.
3. **What changes from ZeRO-2 to ZeRO-3?** Shard persistent parameters, gather for forward and backward, then release; communication rises.
4. **Why does lower memory not imply lower dense FLOPs?** Each data rank computes the whole model on different examples, unlike splitting one matmul with tensor parallelism.
5. **What could invalidate a 30B “fits on GPU” claim?** Activation memory, temporary layer parameter gather, bucket capacity, allocator fragmentation, checkpoint writes and runtime peak.

**Integrity notice:** The explanation and code are agent-assisted. Before asserting personal understanding, inspect the code, reproduce the charts and explain these five questions in your own words. See `README.md` and `ACCURACY_AUDIT.md` for assumptions, limits and sources.

## Submission

The `.ipynb`, source, tests, README, charts, results and accuracy audit are packaged in the companion ZIP. **A GitHub repository has not been published automatically**, so there is no verified live repository URL to share. Push the files to your own GitHub repository and use that real URL when submitting.